# Arousal Prediction — v5 Pipeline

## Key findings and improvements from v4 → v5

| Observation / Issue in v4 | Fix / Improvement in v5 |
|---|---|
| Outliers were detected *before* per-person normalization, causing baseline shifts to be flagged incorrectly. | Swapped steps: Z-score **normalization first**, then IsolationForest outlier detection on the normalized data. |
| RF outperformed LGBM heavily due to the small dataset size (1450 samples). LGBM overfits. | Added **ExtraTreesRegressor**. ExtraTrees shines on small, noisy datasets by randomizing splits, often outperforming RF. |
| Strict rounding `ordinal_clip` and percentile calibration can be sub-optimal for MAE. | Introduced **OOF Threshold Optimization** using Nelder-Mead to find the exact classification boundaries (e.g., 1.6, 2.4, etc.) that minimize MAE on Out-of-Fold predictions. |
| Lag features (1, 2) were powerful, but short-term trends were missing. | Added **Lag-3** and **Rolling-3 window stats** (mean/std) for key features to capture short-term temporal trends. |
| Top 60 features might be too restrictive with an ensemble of 3 models. | Expanded feature selection to **Top 85** features to give ExtraTrees and LGBM more signals to build diverse trees. |

In [1]:
!pip install pandas numpy scipy scikit-learn lightgbm imbalanced-learn matplotlib


[notice] A new release of pip is available: 26.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.signal import welch
from scipy.interpolate import interp1d
import scipy.optimize as opt
from functools import partial
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor, IsolationForest, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error
import lightgbm as lgb
from imblearn.over_sampling import ADASYN, SMOTE
import matplotlib.pyplot as plt

np.random.seed(42)
print('Libraries loaded.')

Libraries loaded.


## 1. Load Data

In [3]:
DATA = '.'

train_labels = pd.read_csv(f'{DATA}/train-label.csv')
test_labels  = pd.read_csv(f'{DATA}/test-label.csv')

trainbvp   = pd.read_csv(f'{DATA}/train-bvp.csv')
traineda   = pd.read_csv(f'{DATA}/train-eda.csv')
traintemp  = pd.read_csv(f'{DATA}/train-temp.csv')
trainhr    = pd.read_csv(f'{DATA}/train-hr.csv')
trainibi   = pd.read_csv(f'{DATA}/train-ibi.csv')
trainbrain = pd.read_csv(f'{DATA}/train-brain.csv')
trainacc   = pd.read_csv(f'{DATA}/train-acc.csv')

testbvp    = pd.read_csv(f'{DATA}/test-bvp.csv')
testeda    = pd.read_csv(f'{DATA}/test-eda.csv')
testtemp   = pd.read_csv(f'{DATA}/test-temp.csv')
testhr     = pd.read_csv(f'{DATA}/test-hr.csv')
testibi    = pd.read_csv(f'{DATA}/test-ibi.csv')
testbrain  = pd.read_csv(f'{DATA}/test-brain.csv')
testacc    = pd.read_csv(f'{DATA}/test-acc.csv')

print(f'Train: {len(train_labels)} | Test: {len(test_labels)}')
print(train_labels['arousal'].value_counts().sort_index())

Train: 1456 | Test: 1496
arousal
1     55
2    430
3    554
4    345
5     72
Name: count, dtype: int64


## 2. Feature Helpers

In [4]:
WINDOW       = 5000
BVP_WINDOW   = 10000   # extended: was 5s (0% coverage)
BRAIN_WINDOW = 10000
IBI_WINDOW   = 20000   # extended: wider HRV window
BASELINE_W   = 30000   # 30s lookback for delta features

def assign_windows_forward(sensor_df, labels_df, window):
    results = []
    for pid in labels_df['pid'].unique():
        lbl = labels_df[labels_df['pid']==pid].sort_values('timestamp')
        sen = sensor_df[sensor_df['pid']==pid].copy()
        if len(sen) == 0:
            continue
        lbl_ts = lbl['timestamp'].values
        bins = np.append(lbl_ts, lbl_ts[-1] + window)
        sen['label_ts'] = pd.cut(sen['timestamp'], bins=bins,
                                  labels=lbl_ts, right=False, include_lowest=True)
        results.append(sen.dropna(subset=['label_ts']))
    if not results:
        return pd.DataFrame()
    out = pd.concat(results, ignore_index=True)
    out['label_ts'] = out['label_ts'].astype(np.int64)
    return out

def assign_windows_lookback(sensor_df, labels_df, window):
    """For each label ts, grab sensor readings from [ts-window, ts)."""
    results = []
    for pid in labels_df['pid'].unique():
        lbl = labels_df[labels_df['pid']==pid].sort_values('timestamp')
        sen = sensor_df[sensor_df['pid']==pid].copy()
        if len(sen) == 0:
            continue
        for ts in lbl['timestamp'].values:
            chunk = sen[(sen['timestamp'] >= ts - window) & (sen['timestamp'] < ts)].copy()
            if len(chunk) > 0:
                chunk['label_ts'] = ts
                chunk['pid'] = pid
                results.append(chunk)
    if not results:
        return pd.DataFrame()
    out = pd.concat(results, ignore_index=True)
    out['label_ts'] = out['label_ts'].astype(np.int64)
    return out

def safe_stats(vals, prefix):
    a = np.array(vals, dtype=float)
    a = a[~np.isnan(a)]
    if len(a) == 0:
        return {f'{prefix}_{k}': np.nan for k in
                ['mean','std','min','max','range','q25','q75','iqr','skew','kurt','rms','count']}
    return {
        f'{prefix}_mean':  float(np.mean(a)),
        f'{prefix}_std':   float(np.std(a)) if len(a)>1 else 0.0,
        f'{prefix}_min':   float(np.min(a)),
        f'{prefix}_max':   float(np.max(a)),
        f'{prefix}_range': float(np.ptp(a)),
        f'{prefix}_q25':   float(np.percentile(a,25)),
        f'{prefix}_q75':   float(np.percentile(a,75)),
        f'{prefix}_iqr':   float(np.percentile(a,75)-np.percentile(a,25)),
        f'{prefix}_skew':  float(stats.skew(a)) if len(a)>2 else 0.0,
        f'{prefix}_kurt':  float(stats.kurtosis(a)) if len(a)>2 else 0.0,
        f'{prefix}_rms':   float(np.sqrt(np.mean(a**2))),
        f'{prefix}_count': float(len(a)),
    }

def spectral_features(vals, fs, prefix):
    a = np.array(vals, dtype=float)
    a = a[~np.isnan(a)]
    if len(a) < 8:
        return {f'{prefix}_lf': np.nan, f'{prefix}_hf': np.nan, f'{prefix}_lf_hf': np.nan}
    try:
        f, psd = welch(a, fs=fs, nperseg=min(len(a), 64))
        lf = np.trapz(psd[(f>=0.04)&(f<=0.15)], f[(f>=0.04)&(f<=0.15)])
        hf = np.trapz(psd[(f>=0.15)&(f<=0.40)], f[(f>=0.15)&(f<=0.40)])
        return {f'{prefix}_lf': float(lf), f'{prefix}_hf': float(hf),
                f'{prefix}_lf_hf': float(lf/(hf+1e-9))}
    except:
        return {f'{prefix}_lf': np.nan, f'{prefix}_hf': np.nan, f'{prefix}_lf_hf': np.nan}

print('Helpers defined.')

Helpers defined.


## 3. Feature Extraction

In [5]:
def build_features(labels_df, bvp_df, eda_df, temp_df, hr_df, ibi_df, brain_df, acc_df):
    print('  Forward windows...')
    bvp_w   = assign_windows_forward(bvp_df,   labels_df, BVP_WINDOW)
    eda_w   = assign_windows_forward(eda_df,   labels_df, WINDOW)
    temp_w  = assign_windows_forward(temp_df,  labels_df, WINDOW)
    hr_w    = assign_windows_forward(hr_df,    labels_df, WINDOW)
    ibi_w   = assign_windows_forward(ibi_df,   labels_df, IBI_WINDOW)
    brain_w = assign_windows_forward(brain_df, labels_df, BRAIN_WINDOW)
    acc_w   = assign_windows_forward(acc_df,   labels_df, WINDOW)

    print('  Lookback baseline windows (30s)...')
    eda_b  = assign_windows_lookback(eda_df,  labels_df, BASELINE_W)
    temp_b = assign_windows_lookback(temp_df, labels_df, BASELINE_W)
    hr_b   = assign_windows_lookback(hr_df,   labels_df, BASELINE_W)
    acc_b  = assign_windows_lookback(acc_df,  labels_df, BASELINE_W)

    # BVP
    print('  BVP...')
    bvp_agg = bvp_w.groupby(['pid','label_ts'])['value'].apply(list).reset_index()
    bvp_feats = []
    for _, r in bvp_agg.iterrows():
        d = {'pid': r['pid'], 'label_ts': r['label_ts']}
        d.update(safe_stats(r['value'], 'bvp'))
        d.update(spectral_features(r['value'], fs=64, prefix='bvp_spec'))
        a = np.array(r['value'])
        d['bvp_zcr'] = float(np.sum(np.diff(np.sign(np.diff(a)))!=0)/len(a)) if len(a)>2 else np.nan
        bvp_feats.append(d)
    bvp_feat_df = pd.DataFrame(bvp_feats)

    # EDA + delta
    print('  EDA...')
    eda_agg = eda_w.groupby(['pid','label_ts'])['value'].apply(list).reset_index()
    eda_b_mean = eda_b.groupby(['pid','label_ts'])['value'].mean().reset_index()
    eda_b_mean.columns = ['pid','label_ts','eda_baseline']
    eda_feats = []
    for _, r in eda_agg.iterrows():
        d = {'pid': r['pid'], 'label_ts': r['label_ts']}
        d.update(safe_stats(r['value'], 'eda'))
        a = np.array(r['value'], dtype=float)
        if len(a) > 2:
            d['eda_slope'] = float(np.polyfit(np.arange(len(a)), a, 1)[0])
            d['eda_peaks'] = float(np.sum((np.diff(np.sign(np.diff(a))))<0))
        else:
            d['eda_slope'] = np.nan; d['eda_peaks'] = np.nan
        eda_feats.append(d)
    eda_feat_df = pd.DataFrame(eda_feats)
    eda_feat_df = eda_feat_df.merge(eda_b_mean, on=['pid','label_ts'], how='left')
    eda_feat_df['eda_delta'] = eda_feat_df['eda_mean'] - eda_feat_df['eda_baseline']

    # TEMP + delta
    print('  TEMP...')
    temp_feat_df = temp_w.groupby(['pid','label_ts'])['value'].agg(
        temp_mean='mean', temp_std='std', temp_min='min',
        temp_max='max', temp_range=lambda x: x.max()-x.min()).reset_index()
    temp_b_mean = temp_b.groupby(['pid','label_ts'])['value'].mean().reset_index()
    temp_b_mean.columns = ['pid','label_ts','temp_baseline']
    temp_feat_df = temp_feat_df.merge(temp_b_mean, on=['pid','label_ts'], how='left')
    temp_feat_df['temp_delta'] = temp_feat_df['temp_mean'] - temp_feat_df['temp_baseline']

    # HR + delta
    print('  HR...')
    hr_feat_df = hr_w.groupby(['pid','label_ts'])['value'].agg(
        hr_mean='mean', hr_std='std', hr_min='min',
        hr_max='max', hr_range=lambda x: x.max()-x.min()).reset_index()
    hr_b_mean = hr_b.groupby(['pid','label_ts'])['value'].mean().reset_index()
    hr_b_mean.columns = ['pid','label_ts','hr_baseline']
    hr_feat_df = hr_feat_df.merge(hr_b_mean, on=['pid','label_ts'], how='left')
    hr_feat_df['hr_delta'] = hr_feat_df['hr_mean'] - hr_feat_df['hr_baseline']

    # IBI + HRV
    print('  IBI/HRV...')
    ibi_agg = ibi_w.groupby(['pid','label_ts'])['value'].apply(list).reset_index()
    ibi_feats = []
    for _, r in ibi_agg.iterrows():
        d = {'pid': r['pid'], 'label_ts': r['label_ts']}
        d.update(safe_stats(r['value'], 'ibi'))
        a = np.array(r['value'], dtype=float)
        a = a[~np.isnan(a)]
        if len(a) > 1:
            diffs = np.diff(a)
            d['ibi_rmssd'] = float(np.sqrt(np.mean(diffs**2)))
            d['ibi_sdnn']  = float(np.std(a))
            d['ibi_pnn50'] = float(np.mean(np.abs(diffs)>50))
        else:
            d['ibi_rmssd'] = np.nan; d['ibi_sdnn'] = np.nan; d['ibi_pnn50'] = np.nan
        d.update(spectral_features(r['value'], fs=4, prefix='ibi_spec'))
        ibi_feats.append(d)
    ibi_feat_df = pd.DataFrame(ibi_feats)

    # Brain
    print('  Brain...')
    brain_cols = ['delta','lowAlpha','highAlpha','lowBeta','highBeta','lowGamma','middleGamma','theta']
    brain_agg = brain_w.groupby(['pid','label_ts'])[brain_cols].agg(['mean','std']).reset_index()
    brain_agg.columns = ['pid','label_ts'] + [f'brain_{c}_{s}' for c in brain_cols for s in ['mean','std']]
    brain_agg['brain_alpha_beta']  = ((brain_agg['brain_lowAlpha_mean']+brain_agg['brain_highAlpha_mean']) /
                                       (brain_agg['brain_lowBeta_mean']+brain_agg['brain_highBeta_mean']+1e-6))
    brain_agg['brain_theta_alpha'] = (brain_agg['brain_theta_mean'] /
                                       (brain_agg['brain_lowAlpha_mean']+brain_agg['brain_highAlpha_mean']+1e-6))
    brain_agg['brain_engagement']  = (brain_agg['brain_lowBeta_mean'] /
                                       (brain_agg['brain_theta_mean']+brain_agg['brain_lowAlpha_mean']+1e-6))

    # ACC + delta
    print('  ACC...')
    acc_w['mag'] = np.sqrt(acc_w['x']**2 + acc_w['y']**2 + acc_w['z']**2)
    acc_b['mag'] = np.sqrt(acc_b['x']**2 + acc_b['y']**2 + acc_b['z']**2)
    acc_agg   = acc_w.groupby(['pid','label_ts'])['mag'].apply(list).reset_index()
    acc_b_mean = acc_b.groupby(['pid','label_ts'])['mag'].mean().reset_index()
    acc_b_mean.columns = ['pid','label_ts','acc_baseline']
    acc_feats = []
    for _, r in acc_agg.iterrows():
        d = {'pid': r['pid'], 'label_ts': r['label_ts']}
        d.update(safe_stats(r['mag'], 'acc'))
        a = np.array(r['mag'], dtype=float)
        if len(a) > 2:
            jerk = np.diff(a)
            d['acc_jerk_mean'] = float(np.mean(np.abs(jerk)))
            d['acc_jerk_std']  = float(np.std(jerk))
        else:
            d['acc_jerk_mean'] = np.nan; d['acc_jerk_std'] = np.nan
        acc_feats.append(d)
    acc_feat_df = pd.DataFrame(acc_feats)
    acc_feat_df = acc_feat_df.merge(acc_b_mean, on=['pid','label_ts'], how='left')
    acc_feat_df['acc_delta'] = acc_feat_df['acc_mean'] - acc_feat_df['acc_baseline']

    # Merge
    print('  Merging...')
    base = labels_df[['id','pid','timestamp']].copy()
    base['label_ts'] = base['timestamp']
    merged = base
    for fdf in [bvp_feat_df, eda_feat_df, temp_feat_df, hr_feat_df,
                ibi_feat_df, brain_agg, acc_feat_df]:
        fdf['label_ts'] = fdf['label_ts'].astype(np.int64)
        merged = merged.merge(fdf, on=['pid','label_ts'], how='left')
    return merged.drop(columns=['label_ts'])

print('build_features() defined.')

build_features() defined.


In [6]:
print('Extracting TRAIN features (~4-6 min)...')
train_feats = build_features(train_labels, trainbvp, traineda, traintemp,
                              trainhr, trainibi, trainbrain, trainacc)
print(f'Train: {train_feats.shape}, missing: {train_feats.isnull().mean().mean():.2%}')

Extracting TRAIN features (~4-6 min)...
  Forward windows...
  Lookback baseline windows (30s)...
  BVP...
  EDA...
  TEMP...
  HR...
  IBI/HRV...
  Brain...
  ACC...
  Merging...
Train: (1456, 102), missing: 19.11%


In [7]:
print('Extracting TEST features (~4-6 min)...')
test_feats = build_features(test_labels, testbvp, testeda, testtemp,
                             testhr, testibi, testbrain, testacc)
print(f'Test: {test_feats.shape}, missing: {test_feats.isnull().mean().mean():.2%}')

Extracting TEST features (~4-6 min)...
  Forward windows...
  Lookback baseline windows (30s)...
  BVP...
  EDA...
  TEMP...
  HR...
  IBI/HRV...
  Brain...
  ACC...
  Merging...
Test: (1496, 102), missing: 19.73%


## 4. Lag & Rolling Features (Expanded for v5)

In [8]:
LAG_COLS = [
    'eda_mean','eda_min','eda_q75','eda_slope','eda_delta',
    'temp_mean','temp_min','temp_delta',
    'hr_mean','hr_std','hr_delta',
    'bvp_mean','bvp_skew',
    'acc_mean','acc_max','acc_delta',
    'ibi_rmssd','ibi_sdnn',
]

def add_lag_features(feat_df, lag_cols, lags=[1, 2, 3]):
    parts = []
    for pid in feat_df['pid'].unique():
        sub = feat_df[feat_df['pid'] == pid].sort_values('timestamp').copy()
        for lag in lags:
            for col in lag_cols:
                if col in sub.columns:
                    sub[f'{col}_lag{lag}'] = sub[col].shift(lag)
        # Add rolling window features
        for col in lag_cols:
            if col in sub.columns:
                sub[f'{col}_roll3_mean'] = sub[col].rolling(3, min_periods=1).mean()
                sub[f'{col}_roll3_std']  = sub[col].rolling(3, min_periods=1).std()
        parts.append(sub)
    return pd.concat(parts).sort_values('id').reset_index(drop=True)

print('Adding lag/rolling features to train...')
train_w_lags = add_lag_features(train_feats, LAG_COLS)
print('Adding lag/rolling features to test...')
test_w_lags  = add_lag_features(test_feats,  LAG_COLS)

print(f'Train shape after lags/rolling: {train_w_lags.shape}')

Adding lag/rolling features to train...
Adding lag/rolling features to test...
Train shape after lags/rolling: (1456, 192)


## 5. Missing Indicators + Imputation

In [9]:
train_df = train_w_lags.merge(train_labels[['id','arousal']], on='id', how='left')
test_df  = test_w_lags.copy()

feat_cols = [c for c in train_df.columns if c not in ['id','pid','timestamp','arousal']]

sensor_groups = {
    'bvp':   [c for c in feat_cols if c.startswith('bvp')],
    'eda':   [c for c in feat_cols if c.startswith('eda')],
    'temp':  [c for c in feat_cols if c.startswith('temp')],
    'hr':    [c for c in feat_cols if c.startswith('hr')],
    'ibi':   [c for c in feat_cols if c.startswith('ibi')],
    'brain': [c for c in feat_cols if c.startswith('brain')],
    'acc':   [c for c in feat_cols if c.startswith('acc')],
}
for grp, cols in sensor_groups.items():
    if cols:
        ind = f'{grp}_missing'
        train_df[ind] = train_df[cols].isnull().any(axis=1).astype(int)
        test_df[ind]  = test_df[cols].isnull().any(axis=1).astype(int)

feat_cols = [c for c in train_df.columns if c not in ['id','pid','timestamp','arousal']]
indicator_cols = [f'{g}_missing' for g in sensor_groups]

def impute_by_pid(df, cols):
    df = df.copy()
    df[cols] = df[cols].fillna(df.groupby('pid')[cols].transform('median'))
    df[cols] = df[cols].fillna(df[cols].median())
    return df

train_df = impute_by_pid(train_df, feat_cols)
test_df  = impute_by_pid(test_df,  feat_cols)
test_df[feat_cols] = test_df[feat_cols].fillna(train_df[feat_cols].median())

print(f'Total feature count: {len(feat_cols)}')
print(f'Train NaN: {train_df[feat_cols].isnull().sum().sum()}')
print(f'Test  NaN: {test_df[feat_cols].isnull().sum().sum()}')

Total feature count: 196
Train NaN: 8736
Test  NaN: 8976


## 6. Per-Person Normalisation (Moved BEFORE Outlier Detection)

In [10]:
norm_cols = [c for c in feat_cols if not c.endswith('_missing')]

def zscore_by_pid(df, cols):
    df = df.copy()
    for pid in df['pid'].unique():
        mask = df['pid'] == pid
        sub  = df.loc[mask, cols]
        mu   = sub.mean()
        sig  = sub.std().replace(0, np.nan)
        df.loc[mask, cols] = (sub - mu) / sig
    df[cols] = df[cols].fillna(0)
    return df

train_norm = zscore_by_pid(train_df, norm_cols)
test_norm  = zscore_by_pid(test_df,  norm_cols)
print('Per-pid normalisation done.')
print(f'Train NaN: {train_norm[feat_cols].isnull().sum().sum()}')

Per-pid normalisation done.
Train NaN: 0


## 7. Outlier Detection (Run on normalized space)

In [11]:
X_all = train_norm[feat_cols].values
y_all = train_norm['arousal'].values
outlier_mask = np.zeros(len(train_norm), dtype=bool)

for cls in np.unique(y_all):
    idx = np.where(y_all == cls)[0]
    # Be slightly more conservative on normalized data
    contamination = 0.03 if len(idx) < 100 else 0.05
    iso = IsolationForest(n_estimators=100, contamination=contamination, random_state=42)
    preds = iso.fit_predict(X_all[idx])
    n_out = np.sum(preds == -1)
    print(f'  Class {cls} ({len(idx)}): {n_out} outliers ({n_out/len(idx):.1%})')
    outlier_mask[idx[preds==-1]] = True

train_clean = train_norm[~outlier_mask].copy().reset_index(drop=True)
print(f'Kept: {len(train_clean)} / {len(train_norm)}')
print(train_clean['arousal'].value_counts().sort_index())

  Class 1 (55): 2 outliers (3.6%)
  Class 2 (430): 22 outliers (5.1%)
  Class 3 (554): 28 outliers (5.1%)
  Class 4 (345): 18 outliers (5.2%)
  Class 5 (72): 3 outliers (4.2%)
Kept: 1383 / 1456
arousal
1     53
2    408
3    526
4    327
5     69
Name: count, dtype: int64


## 8. Feature Selection — top 85

In [12]:
X_sel = train_clean[feat_cols].values
y_sel = train_clean['arousal'].values.astype(float)

rf_sel = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf_sel.fit(X_sel, y_sel)

imp_df = pd.DataFrame({'feature': feat_cols,
                        'importance': rf_sel.feature_importances_,
                        'type': ['lag/roll' if ('_lag' in c or '_roll' in c) else 'delta' if '_delta' in c else 'base'
                                 for c in feat_cols]
                       }).sort_values('importance', ascending=False)

N_KEEP = 85
top_features = imp_df.head(N_KEEP)['feature'].tolist()
# ensure indicators are included
for ind in indicator_cols:
    if ind not in top_features and ind in feat_cols:
        top_features.append(ind)

print(f'Selected {len(top_features)} features')
print('\nTop 25 features:')
print(imp_df.head(25)[['feature','importance','type']].to_string(index=False))

Selected 92 features

Top 25 features:
            feature  importance     type
 eda_min_roll3_mean    0.081535 lag/roll
          acc_count    0.070629     base
          bvp_count    0.070245     base
          eda_count    0.060787     base
           ibi_skew    0.030965     base
          ibi_count    0.017525     base
        hr_baseline    0.016023     base
       hr_mean_lag3    0.013427 lag/roll
       hr_mean_lag2    0.013322 lag/roll
      temp_baseline    0.012349     base
       acc_baseline    0.012247     base
       eda_min_lag1    0.011400 lag/roll
      hr_delta_lag3    0.011378 lag/roll
            eda_min    0.010843     base
           hr_delta    0.010533    delta
      hr_delta_lag2    0.010176 lag/roll
       eda_min_lag2    0.009531 lag/roll
      eda_mean_lag2    0.009376 lag/roll
       eda_baseline    0.008739     base
 bvp_mean_roll3_std    0.008578 lag/roll
 ibi_sdnn_roll3_std    0.008264 lag/roll
            bvp_q75    0.008001     base
      temp_min_lag

## 9. ADASYN Oversampling

In [13]:
X_train = train_clean[top_features].values
y_train = train_clean['arousal'].values

unique, counts = np.unique(y_train, return_counts=True)
print('Before resampling:')
for u, c in zip(unique, counts): print(f'  Class {u}: {c}')

target = {cls: max(cnt, 150) for cls, cnt in zip(unique.astype(int), counts)}
try:
    res = ADASYN(sampling_strategy=target, n_neighbors=min(4, min(counts)-1), random_state=42)
    X_res, y_res = res.fit_resample(X_train, y_train)
    print('ADASYN succeeded.')
except Exception as e:
    print(f'ADASYN failed ({e}), using SMOTE...')
    res = SMOTE(sampling_strategy=target, k_neighbors=min(3, min(counts)-1), random_state=42)
    X_res, y_res = res.fit_resample(X_train, y_train)

print('After resampling:')
u2, c2 = np.unique(y_res, return_counts=True)
for u, c in zip(u2, c2): print(f'  Class {u}: {c}')

# Class weights — 2× boost for extremes (classes 1 and 5)
class_counts_res = dict(zip(*np.unique(y_res, return_counts=True)))
total_res = len(y_res)
class_weights = {cls: total_res/(len(class_counts_res)*cnt) for cls,cnt in class_counts_res.items()}
class_weights[1] = class_weights[1] * 2.0
class_weights[5] = class_weights[5] * 2.0
sample_weights = np.array([class_weights[y] for y in y_res])
print('\nClass weights:')
for k,v in sorted(class_weights.items()): print(f'  Class {k}: {v:.3f}')

Before resampling:
  Class 1: 53
  Class 2: 408
  Class 3: 526
  Class 4: 327
  Class 5: 69
ADASYN succeeded.
After resampling:
  Class 1: 146
  Class 2: 408
  Class 3: 526
  Class 4: 327
  Class 5: 143

Class weights:
  Class 1: 4.247
  Class 2: 0.760
  Class 3: 0.589
  Class 4: 0.948
  Class 5: 4.336


## 10. Leave-One-Pid-Out CV & Threshold Optimization

In [14]:
rf_params = dict(
    n_estimators=500, max_depth=10, min_samples_leaf=4,
    min_samples_split=8, max_features='sqrt', max_samples=0.8,
    random_state=42, n_jobs=-1
)
et_params = dict(
    n_estimators=500, max_depth=12, min_samples_leaf=2,
    min_samples_split=6, max_features='sqrt', max_samples=0.8,
    bootstrap=True, random_state=42, n_jobs=-1
)
lgb_params = dict(
    objective='regression_l1', metric='mae',
    n_estimators=400, learning_rate=0.03, num_leaves=31, max_depth=5,
    min_child_samples=20, subsample=0.75, colsample_bytree=0.75,
    reg_alpha=0.5, reg_lambda=1.0, random_state=42, verbose=-1
)

pids = train_clean['pid'].unique()
lopo_maes = {'rf': [], 'et': [], 'lgb': [], 'ens': []}
lopo_pid_results = []

oof_preds = np.zeros(len(train_clean))
oof_y = train_clean['arousal'].values

print(f'LOPO CV ({len(pids)} folds)...\n')

for held_pid in pids:
    mask_val = train_clean['pid'] == held_pid
    mask_tr  = ~mask_val
    X_tr  = train_clean.loc[mask_tr,  top_features].values
    y_tr  = train_clean.loc[mask_tr,  'arousal'].values.astype(float)
    X_val = train_clean.loc[mask_val, top_features].values
    y_val = train_clean.loc[mask_val, 'arousal'].values

    u_tr, c_tr = np.unique(y_tr, return_counts=True)
    fold_target = {cls: max(cnt, 100) for cls,cnt in zip(u_tr.astype(int), c_tr)}
    safe_k = max(1, min(3, min(c_tr)-1))
    try:
        rf_sm = ADASYN(sampling_strategy=fold_target, n_neighbors=safe_k, random_state=42)
        X_tr_s, y_tr_s = rf_sm.fit_resample(X_tr, y_tr)
    except:
        sm = SMOTE(sampling_strategy=fold_target, k_neighbors=safe_k, random_state=42)
        X_tr_s, y_tr_s = sm.fit_resample(X_tr, y_tr)

    sw = np.array([class_weights.get(int(y), 1.0) for y in y_tr_s])

    # 1. RF
    m_rf = RandomForestRegressor(**rf_params)
    m_rf.fit(X_tr_s, y_tr_s, sample_weight=sw)
    p_rf_raw = m_rf.predict(X_val)

    # 2. ExtraTrees
    m_et = ExtraTreesRegressor(**et_params)
    m_et.fit(X_tr_s, y_tr_s, sample_weight=sw)
    p_et_raw = m_et.predict(X_val)

    # 3. LGB
    m_lgb = lgb.LGBMRegressor(**lgb_params)
    m_lgb.fit(X_tr_s, y_tr_s, sample_weight=sw)
    p_lgb_raw = m_lgb.predict(X_val)
    
    # Simple unweighted ensemble for RAW predictions (can be tuned later)
    p_ens_raw = 0.4 * p_rf_raw + 0.4 * p_et_raw + 0.2 * p_lgb_raw
    oof_preds[mask_val] = p_ens_raw
    
    def simple_clip(p): return np.clip(np.round(p), 1, 5).astype(int)

    mae_rf  = mean_absolute_error(y_val, simple_clip(p_rf_raw))
    mae_et  = mean_absolute_error(y_val, simple_clip(p_et_raw))
    mae_lgb = mean_absolute_error(y_val, simple_clip(p_lgb_raw))
    mae_ens = mean_absolute_error(y_val, simple_clip(p_ens_raw))

    lopo_maes['rf'].append(mae_rf)
    lopo_maes['et'].append(mae_et)
    lopo_maes['lgb'].append(mae_lgb)
    lopo_maes['ens'].append(mae_ens)
    lopo_pid_results.append({'pid': held_pid, 'n': mask_val.sum(),
                              'rf': mae_rf, 'et': mae_et, 'lgb': mae_lgb, 'ens': mae_ens})

    print(f'  [{held_pid}] RF={mae_rf:.4f} ET={mae_et:.4f} LGB={mae_lgb:.4f} ENS={mae_ens:.4f}')

print('\nLOPO Summary (Before Threshold Optimization):')
for name, maes in lopo_maes.items():
    print(f'{name.upper():6s} LOPO MAE: {np.mean(maes):.4f} ± {np.std(maes):.4f}')

LOPO CV (11 folds)...

  [01Z2] RF=0.8661 ET=0.7500 LGB=1.1518 ENS=0.8839
  [70N8] RF=0.4718 ET=0.4859 LGB=0.5352 ENS=0.4930
  [7PF3] RF=0.7174 ET=0.8696 LGB=0.7826 ENS=0.7826
  [CQ2G] RF=1.0256 ET=1.0256 LGB=1.1624 ENS=1.0342
  [D1XP] RF=0.2706 ET=0.2706 LGB=0.3471 ENS=0.2412
  [DT5C] RF=1.1217 ET=0.9130 LGB=1.2957 ENS=1.1826
  [F1ZM] RF=0.8174 ET=0.8261 LGB=0.9043 ENS=0.8174
  [LIUY] RF=1.4344 ET=0.8852 LGB=1.7459 ENS=1.3197
  [SE4Q] RF=0.6261 ET=0.6435 LGB=0.9043 ENS=0.6870
  [TPQI] RF=0.7632 ET=0.7719 LGB=0.9825 ENS=0.7632
  [Y21H] RF=0.8618 ET=0.8537 LGB=0.9675 ENS=0.8862

LOPO Summary (Before Threshold Optimization):
RF     LOPO MAE: 0.8160 ± 0.2998
ET     LOPO MAE: 0.7541 ± 0.2051
LGB    LOPO MAE: 0.9799 ± 0.3565
ENS    LOPO MAE: 0.8264 ± 0.2859


## 11. Optimize Classification Thresholds on OOF Predictions

In [15]:
def get_class_bounds(preds, bounds):
    p = np.copy(preds)
    p = np.where(p < bounds[0], 1,
        np.where(p < bounds[1], 2,
        np.where(p < bounds[2], 3,
        np.where(p < bounds[3], 4, 5))))
    return p

def mae_loss(bounds, preds, y):
    return mean_absolute_error(y, get_class_bounds(preds, bounds))

init_bounds = [1.5, 2.5, 3.5, 4.5]
res = opt.minimize(mae_loss, init_bounds, args=(oof_preds, oof_y), method='Nelder-Mead')
opt_bounds = res.x
opt_bounds.sort()

print(f"Default Thresholds: {init_bounds}")
print(f"Default MAE:        {mae_loss(init_bounds, oof_preds, oof_y):.4f}")
print(f"Optimized Bounds:   {opt_bounds}")
print(f"Optimized OOF MAE:  {mae_loss(opt_bounds, oof_preds, oof_y):.4f}")

Default Thresholds: [1.5, 2.5, 3.5, 4.5]
Default MAE:        0.7990
Optimized Bounds:   [1.40756035 2.36222458 4.14155006 4.17767143]
Optimized OOF MAE:  0.7101


## 12. Final Training & Prediction

In [16]:
print('Training final models...')

m_rf_final = RandomForestRegressor(**rf_params)
m_rf_final.fit(X_res, y_res.astype(float), sample_weight=sample_weights)
print('  RF done.')

m_et_final = ExtraTreesRegressor(**et_params)
m_et_final.fit(X_res, y_res.astype(float), sample_weight=sample_weights)
print('  ET done.')

m_lgb_final = lgb.LGBMRegressor(**lgb_params)
m_lgb_final.fit(X_res, y_res.astype(float), sample_weight=sample_weights)
print('  LGB done.')

# Test predictions
X_test = test_norm[top_features].values
raw_rf  = m_rf_final.predict(X_test)
raw_et  = m_et_final.predict(X_test)
raw_lgb = m_lgb_final.predict(X_test)

raw_ens_test = 0.4 * raw_rf + 0.4 * raw_et + 0.2 * raw_lgb

# Apply Optimized Bounds
pred_classes = get_class_bounds(raw_ens_test, opt_bounds)

print('\nPrediction distribution (after optimization):')
u, c = np.unique(pred_classes, return_counts=True)
for cls, cnt in zip(u, c):
    pct = cnt/len(pred_classes)*100
    train_pct = (train_labels['arousal']==cls).mean()*100
    print(f'  Class {cls}: {cnt:4d} ({pct:.1f}%) | Train {train_pct:.1f}% | Δ={pct-train_pct:+.1f}%')

Training final models...
  RF done.
  ET done.
  LGB done.

Prediction distribution (after optimization):
  Class 2:   63 (4.2%) | Train 29.5% | Δ=-25.3%
  Class 3: 1433 (95.8%) | Train 38.0% | Δ=+57.7%


## 13. Generate Submission

In [17]:
submission = pd.DataFrame({'id': test_labels['id'].values, 'arousal': pred_classes})

print('Final distribution:')
print(submission['arousal'].value_counts().sort_index())
submission.to_csv('submission_v5.csv', index=False)
print('\nSaved: submission_v5.csv')

print('\n=== v4 vs v5 Summary ===')
print(f'v4 ENS LOPO MAE: 0.8002')
print(f'v5 ENS LOPO MAE (Optimized): {mae_loss(opt_bounds, oof_preds, oof_y):.4f}')

Final distribution:
arousal
2      63
3    1433
Name: count, dtype: int64

Saved: submission_v5.csv

=== v4 vs v5 Summary ===
v4 ENS LOPO MAE: 0.8002
v5 ENS LOPO MAE (Optimized): 0.7101
